# Step 5a: RAG Framework — Indexing & Vector Database
## YouTube Fast Fashion Intelligence Engine — CSCI370

### What is RAG?

**RAG = Retrieval Augmented Generation**

It is a technique that combines two things:
1. **Retrieval** — searching a database of your comments to find the most relevant ones
2. **Generation** — feeding those relevant comments to an LLM to generate a grounded answer

Without RAG, if you ask an LLM *"What do people say about Shein's labor practices?"*
it would answer from its training data — which may be outdated or hallucinated.

With RAG, it:
1. Searches YOUR 30,609 comments for the most relevant ones
2. Shows those real comments to the LLM
3. The LLM answers based only on what real people actually said

This makes answers **accurate, specific, and grounded in your data.**

### What this notebook does (Step 5a — Indexing)

This notebook runs **once** to build the searchable database.
It:
- Converts all comments into embeddings (vectors)
- Stores them in **ChromaDB** (a vector database) with all metadata
- Builds a **BM25 index** for lexical (keyword) search
- Saves everything to disk so Step 5b can load and search it instantly

Think of this as building the library. Step 5b is using the library.


## 0. Install Libraries

In [1]:
!pip install chromadb rank_bm25 groq scikit-learn -q


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load Libraries and Data

In [2]:
import pandas as pd
import numpy as np
import chromadb
from chromadb import EmbeddingFunction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from rank_bm25 import BM25Okapi
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Load the final dataset — output of Step 4 (topics + sentiment + NER)
# If you haven't run Step 4 yet, use youtube_comments_english.csv instead
try:
    df = pd.read_csv('youtube_comments_topics.csv')
    print("Loaded topics dataset")
except FileNotFoundError:
    df = pd.read_csv('youtube_comments_english.csv')
    print("Topics file not found — loaded english dataset instead")

df = df.dropna(subset=['text_clean']).reset_index(drop=True)

print(f"Total comments: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Loaded topics dataset
Total comments: 30,609
Columns: ['author', 'updated_at', 'like_count', 'text', 'video_id', 'public', 'text_clean', 'sentiment_score', 'sentiment_label', 'sentiment_source', 'language', 'topic_general', 'topic_general_label', 'topic_positive', 'topic_negative', 'topic_neutral']


,author,updated_at,like_count,text,video_id,public,text_clean,sentiment_score,sentiment_label,sentiment_source,language,topic_general,topic_general_label,topic_positive,topic_negative,topic_neutral
0,@joaquinrodriguezvillegas366,2026-02-25 00:49:20+00:00,0,This doesn't surprise me since the entire hist...,qpClEvsjW0s,True,This doesn't surprise me since the entire hist...,-0.0121,Negative,RoBERTa,en,0,Fast Fashion Shopping Habits,-1,4,-1
1,@andrerosekriel1127,2026-02-22 11:15:55+00:00,0,Okay then why work for these factories.....or ...,qpClEvsjW0s,True,Okay then why work for these factories.....or ...,-0.2732,Negative,VADER,en,0,Fast Fashion Shopping Habits,-1,0,-1
2,@sbradshaw1886,2026-03-17 02:12:15+00:00,0,Please do not be this ignorant. The US and oth...,qpClEvsjW0s,True,Please do not be this ignorant. The US and oth...,-0.0194,Negative,RoBERTa,en,0,Fast Fashion Shopping Habits,-1,2,-1


## 2. Build the Embedding Function

**What is an embedding?**

An embedding converts text into a list of numbers (a vector) that captures its *meaning*.
Comments that talk about similar things will have similar vectors — even if they use different words.

For example:
- *"Shein workers are paid almost nothing"* 
- *"Employees at fast fashion brands earn very low wages"*

These say different things with different words, but their embeddings will be close together
because they mean the same thing. This is what enables **semantic search** — finding similar
comments by meaning, not just matching keywords.

We use **TF-IDF + SVD** as our embedding method:
- TF-IDF converts each comment into word frequency scores
- SVD compresses those scores into 128 numbers (dimensions)
- The result is a 128-dimensional vector per comment

We fit the model on ALL comments first, then use it to embed any new query text.


In [3]:
class TFIDFEmbeddingFunction(EmbeddingFunction):
    """
    Custom embedding function for ChromaDB.
    Uses TF-IDF + SVD to convert text into vectors.
    Works fully offline — no HuggingFace downloads needed.

    If you have internet access, you can swap this for:
    from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
    ef = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
    """
    def __init__(self, fit_docs, n_components=128):
        self.tfidf = TfidfVectorizer(
            max_features=8000,
            stop_words='english',
            ngram_range=(1, 2),
            min_df=2
        )
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        # Fit on all documents so the vocabulary covers the full dataset
        X = self.tfidf.fit_transform(fit_docs)
        self.svd.fit(X)
        print(f"Embedding model fitted on {len(fit_docs):,} documents")
        print(f"Vocabulary size: {len(self.tfidf.vocabulary_):,} terms")
        print(f"Embedding dimensions: {n_components}")

    def __call__(self, input):
        """Convert a list of texts into a list of embedding vectors."""
        X = self.tfidf.transform(input)
        return self.svd.transform(X).tolist()

# Fit the embedding model on all comments
all_docs = df['text_clean'].tolist()
embedding_fn = TFIDFEmbeddingFunction(all_docs, n_components=128)

# Save the embedding function for use in Step 5b
with open('embedding_function.pkl', 'wb') as f:
    pickle.dump(embedding_fn, f)
print("\nEmbedding function saved to embedding_function.pkl")

Embedding model fitted on 30,609 documents
Vocabulary size: 8,000 terms
Embedding dimensions: 128

Embedding function saved to embedding_function.pkl


## 3. Build the ChromaDB Vector Database

**What is ChromaDB?**

ChromaDB is a **vector database** — a database designed specifically for storing
and searching embeddings (vectors).

Unlike a regular database where you search by exact match (e.g. WHERE sentiment = 'Negative'),
ChromaDB lets you search by *similarity* — find all comments whose meaning is close to
a query like *"what do people think about Shein's quality?"*

We store each comment with:
- The comment text itself
- Its embedding vector (128 numbers)
- Metadata: sentiment, video_id, topic, like_count, date

The metadata enables **metadata filtering** — e.g. search only within negative comments,
or only comments from a specific video.


In [4]:
# Create a persistent ChromaDB database
# persistent_client saves to disk so you don't have to rebuild every time

DB_PATH = './chroma_db'
os.makedirs(DB_PATH, exist_ok=True)

client = chromadb.PersistentClient(path=DB_PATH)

# Delete existing collection if rebuilding
try:
    client.delete_collection('youtube_comments')
    print("Deleted existing collection")
except:
    pass

# Create the collection with our custom embedding function
collection = client.get_or_create_collection(
    name='youtube_comments',
    embedding_function=embedding_fn,
    metadata={'description': 'YouTube fast fashion comments with sentiment and topics'}
)

print(f"Collection created: {collection.name}")

Collection created: youtube_comments


In [5]:
# Index all comments into ChromaDB in batches
# We use batches of 500 to avoid memory issues with large datasets

BATCH_SIZE = 500
total = len(df)

print(f"Indexing {total:,} comments into ChromaDB...")
print(f"Batch size: {BATCH_SIZE} | Total batches: {total // BATCH_SIZE + 1}")

for start in range(0, total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total)
    batch = df.iloc[start:end]

    # Each document needs a unique ID
    ids = [str(i) for i in batch.index.tolist()]

    # The text to embed and store
    documents = batch['text_clean'].tolist()

    # Metadata stored alongside each comment — used for filtering
    metadatas = []
    for _, row in batch.iterrows():
        meta = {
            'author'          : str(row.get('author', '')),
            'video_id'        : str(row.get('video_id', '')),
            'like_count'      : int(row.get('like_count', 0)),
            'sentiment_label' : str(row.get('sentiment_label', 'Unknown')),
            'sentiment_score' : float(row.get('sentiment_score', 0.0)),
            'updated_at'      : str(row.get('updated_at', '')),
            'topic_general'   : int(row.get('topic_general', -1)) if 'topic_general' in row else -1,
        }
        metadatas.append(meta)

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )

    if (start // BATCH_SIZE) % 5 == 0:
        print(f"  Indexed {end:,} / {total:,} comments...")

print(f"\nDone! Total indexed: {collection.count():,} comments")

Indexing 30,609 comments into ChromaDB...
Batch size: 500 | Total batches: 62
  Indexed 500 / 30,609 comments...
  Indexed 3,000 / 30,609 comments...
  Indexed 5,500 / 30,609 comments...
  Indexed 8,000 / 30,609 comments...
  Indexed 10,500 / 30,609 comments...
  Indexed 13,000 / 30,609 comments...
  Indexed 15,500 / 30,609 comments...
  Indexed 18,000 / 30,609 comments...
  Indexed 20,500 / 30,609 comments...
  Indexed 23,000 / 30,609 comments...
  Indexed 25,500 / 30,609 comments...
  Indexed 28,000 / 30,609 comments...
  Indexed 30,500 / 30,609 comments...

Done! Total indexed: 30,609 comments


## 4. Build the BM25 Index (for Lexical Search)

**What is BM25?**

BM25 (Best Match 25) is a classic keyword search algorithm — it finds documents
that contain the exact words in your query, ranked by how often and how uniquely
those words appear.

BM25 is the backbone of search engines like Elasticsearch and early Google.

**Why do we need BM25 if we already have ChromaDB?**

ChromaDB does *semantic* search (by meaning).
BM25 does *lexical* search (by exact keywords).

They complement each other:
- Semantic: *"labor exploitation"* finds comments about *"workers being underpaid"* even without exact match
- Lexical: *"Shein"* finds every comment containing the exact word "Shein"

We combine both in the **hybrid retrieval** (Step 5b).


In [6]:
# Build BM25 index
# BM25 works on tokenized (split into words) documents

print("Building BM25 index...")

# Tokenize all comments — split into lowercase words
tokenized_corpus = [str(text).lower().split() for text in df['text_clean'].tolist()]

# Build the BM25 index
bm25_index = BM25Okapi(tokenized_corpus)

# Save the BM25 index and the original texts for retrieval
bm25_data = {
    'index'  : bm25_index,
    'texts'  : df['text_clean'].tolist(),
    'indices': df.index.tolist()
}

with open('bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25_data, f)

print(f"BM25 index built and saved!")
print(f"Corpus size: {len(tokenized_corpus):,} documents")

Building BM25 index...
BM25 index built and saved!
Corpus size: 30,609 documents


## 5. Verify the Database Works

Before moving to Step 5b, let's test that both indices return sensible results.


In [7]:
# ── Test 1: Semantic search via ChromaDB ──────────────────────────────────────
print("TEST 1: Semantic Search")
print("-" * 60)

query = "workers being exploited in fast fashion factories"
results = collection.query(
    query_texts=[query],
    n_results=3
)

print(f"Query: '{query}'")
print(f"\nTop 3 semantically similar comments:")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"\n  [{i+1}] Sentiment: {meta['sentiment_label']} | Likes: {meta['like_count']}")
    print(f"       {doc[:200]}")

TEST 1: Semantic Search
------------------------------------------------------------
Query: 'workers being exploited in fast fashion factories'

Top 3 semantically similar comments:

  [1] Sentiment: Negative | Likes: 1
       This is the dark side of fast fashion. I am a Bangladeshi.. Garments workers get paid low salary. They don’t get enough privileges. This is truly injustice to them.

  [2] Sentiment: Negative | Likes: 0
       This makes me cry, on the other hand people on tiktok consuming fast fashion and increasing the demand of such factories

  [3] Sentiment: Negative | Likes: 2
       You who contribute to fast fashion should be ashamed!


In [8]:
# ── Test 2: Metadata filtering ────────────────────────────────────────────────
print("\nTEST 2: Metadata Filtering (Negative comments only)")
print("-" * 60)

results = collection.query(
    query_texts=["shein quality problems"],
    n_results=3,
    where={"sentiment_label": "Negative"}   # filter BEFORE searching
)

print("Query: 'shein quality problems' — filtered to NEGATIVE only")
print(f"\nTop 3 results:")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"\n  [{i+1}] Sentiment: {meta['sentiment_label']}")
    print(f"       {doc[:200]}")


TEST 2: Metadata Filtering (Negative comments only)
------------------------------------------------------------
Query: 'shein quality problems' — filtered to NEGATIVE only

Top 3 results:

  [1] Sentiment: Negative
       Never buy in Shein. Never will. Quality hideous, explotaition of human beings unacceptable.

  [2] Sentiment: Negative
       shein sells pure garbage clothing. the lowest possible quality sold via fake photos

  [3] Sentiment: Negative
       Shein is crazily EXPENSIVE in my county, and the quality is the worst 😭😭😭


In [9]:
# ── Test 3: BM25 lexical search ───────────────────────────────────────────────
print("\nTEST 3: Lexical Search (BM25)")
print("-" * 60)

query_tokens = "shein temu cheap labor".split()
scores = bm25_index.get_scores(query_tokens)
top_indices = np.argsort(scores)[::-1][:3]

print(f"Query tokens: {query_tokens}")
print(f"\nTop 3 BM25 matches:")
for rank, idx in enumerate(top_indices):
    print(f"\n  [{rank+1}] BM25 score: {scores[idx]:.3f}")
    print(f"       {df.iloc[idx]['text_clean'][:200]}")


TEST 3: Lexical Search (BM25)
------------------------------------------------------------
Query tokens: ['shein', 'temu', 'cheap', 'labor']

Top 3 BM25 matches:

  [1] BM25 score: 13.929
       I don't buy from Shein nor Temu both are slave markets for cheap made products.

  [2] BM25 score: 11.882
       Boycott shein and temu

  [3] BM25 score: 11.803
       @LinaB33it’s fine. We need cheap labor


## 6. Summary

**What we built in this notebook:**

| Component | File saved | Purpose |
|---|---|---|
| Embedding function | `embedding_function.pkl` | Convert text → vectors |
| ChromaDB database | `./chroma_db/` folder | Semantic + metadata search |
| BM25 index | `bm25_index.pkl` | Lexical keyword search |

**Run this notebook only once.** Everything is saved to disk.
Step 5b loads these files and uses them for all retrieval and Q&A.

**Next → Step 5b: Retrieval, LLM Q&A, and Agent Orchestration**
